In [1]:
import astropy
import astroquery.simbad
import numpy as np
from astropy.table import Table
from astropy.io import ascii
import astropy.units as u
from astropy.coordinates import SkyCoord
import pyarrow
import pyvo
from astroquery.simbad import Simbad
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
from collections import Counter
import timeit
import matplotlib.pyplot as plt
import pandas
import os


simbad = pyvo.dal.TAPService("https://simbad.cds.unistra.fr/simbad/sim-tap")
nasa = pyvo.dal.TAPService("https://exoplanetarchive.ipac.caltech.edu/TAP")

In [2]:
input = ascii.read("./input/HPIC_LC4_combined_d50.txt")
# input["star_name"][0] = "012345678901234567890123456789"
name_list = input["star_name"]

In [3]:
if os.path.exists("./alternate_ids_aggr_full.txt"):
    alternate_ids_aggr = Table.read("alternate_ids_aggr_full.txt", format="ascii")
else:
    alternate_ids_aggr = simbad.run_sync("""
	SELECT my_ids, ids FROM my_ids LEFT JOIN ident ON my_ids.my_ids = ident.id LEFT JOIN ids USING(oidref)
	""",
										uploads={"my_ids": Table({"my_ids": name_list})}
										).to_table()
    # takes around 1m30s
    alternate_ids_aggr.write("./alternate_ids_aggr_full.txt", format="ascii")

In [4]:
# "basteln" columns with each my_id, id to be able to perform join with NEA
my_ids = []
all_ids = []
for my_id, alt_ids in alternate_ids_aggr.iterrows():
	for id in str(alt_ids).split("|"):
		my_ids.append(my_id)
		all_ids.append(id)
		# all_ids.append(id.replace("'", "''")) # in ADQL, escape single quote ' with ''

alternate_ids_single = Table([my_ids, all_ids], names=["my_ids", "id"], dtype=["str", "str"])
alternate_ids_single
# takes around 1s
# has >270'000 entries!

my_ids,id
str29,str33
TIC 459832522,TIC 459832522
TIC 459832522,AP J14153968+1910558
TIC 459832522,GALAH 150210005801171
TIC 459832522,GJ 541
TIC 459832522,HIP 69673
TIC 459832522,PLX 3242
TIC 459832522,LSPM J1415+1910
TIC 459832522,NLTT 36756
TIC 459832522,TYC 1472-1436-1


In [ ]:
if os.path.exists("./pscomppars.txt"):
    pscomppars = Table.read("pscomppars.txt", format='ascii')
else:
    pscomppars = nasa.search("""
SELECT * FROM pscomppars
    """).to_table()
    # grabbing entire pscomppars takes ~2 minutes
    pscomppars.write("./pscomppars.txt", format='ascii')

In [18]:
my_pscomppars = pscomppars.to_table().copy()
pd_pscomppars = my_pscomppars.to_pandas()

In [21]:
df_alt_ids = alternate_ids_single.to_pandas()

In [26]:
# all_in_my_ids = pd_pscomppars[pd_pscomppars["hostname"].isin(alternate_ids_single["id"])]
# len(all_in_my_ids)

469

In [42]:
final = pd_pscomppars.merge(
    df_alt_ids,
    left_on="hostname",
    right_on="id",
    how="inner"
)
# final.drop(columns="id")

In [41]:
final[["pl_name", "sy_dist", "hostname", "id", "my_ids"]]

,pl_name,sy_dist,hostname,id,my_ids
0,Kepler-1167 b,820.905,Kepler-1167,NaN,NaN
1,Kepler-1740 b,1061.770,Kepler-1740,NaN,NaN
2,Kepler-1581 b,493.175,Kepler-1581,NaN,NaN
3,Kepler-644 b,1318.050,Kepler-644,NaN,NaN
4,Kepler-1752 b,962.888,Kepler-1752,NaN,NaN
...,...,...,...,...,...
6104,NGTS-34 b,120.475,NGTS-34,NaN,NaN
6105,OGLE-TR-113 b,566.459,OGLE-TR-113,NaN,NaN
6106,Kepler-76 b,822.960,Kepler-76,NaN,NaN
6107,WASP-47 c,264.780,WASP-47,NaN,NaN


In [39]:
print(nasa.search("SELECT COUNT(*) FROM pscomppars WHERE sy_dist < 50").to_table()["count(*)"][0])
print(nasa.search("SELECT COUNT(*) FROM pscomppars WHERE sy_dist < 50 AND st_teff < 6000 AND st_teff > 3000").to_table()["count(*)"][0])

929
830


In [58]:
count 
for i, hostname in enumerate(my_pscomppars["hostname"]):
	if hostname not in alternate_ids_single["id"]:
		print(hostname)

Kepler-1167
Kepler-1740
Kepler-1581
Kepler-644
Kepler-1752
Kepler-280
Kepler-1208
Kepler-263
Kepler-1101
K2-19
Kepler-560
Kepler-150
Kepler-498
Kepler-817
Kepler-937
K2-10
Kepler-571
Kepler-1458
K2-335
Kepler-1600
Kepler-221
Kepler-626
Kepler-1549
K2-62
Kepler-1149
Kepler-1288
Kepler-128
Kepler-1245
Kepler-1068
Kepler-311
Kepler-150
Kepler-272
Kepler-152
Kepler-1506
CoRoT-3
HD 205739
NGTS-10
HD 224693
HD 12661
2MASS J12073346-3932539
HD 16417
HD 19994
GQ Lup
HD 37605
HD 33283
HD 9446
HD 110014
HD 79498
HD 45652
HD 3167
HD 47186
HD 148156
HD 12661
HIP 14810
HIP 14810
gam1 Leo
HD 156411
HD 8535
HD 37605
HD 1605
BD+20 2457
HD 240210
CoRoT-17
HD 132563
HD 55696
HD 148164
24 Sex
HD 22781
HD 68988
HD 148164
HD 29985
HD 3167
HD 90156
HD 45350
MOA-2009-BLG-266L
Kepler-595
Kepler-595
HD 177830
OGLE-TR-111
PSR B1257+12
PSR B1257+12
MOA-2007-BLG-192L
CoRoT-18
Kepler-45
WASP-23
SWEEPS-11
OGLE-TR-211
Lupus-TR-3
OGLE-2005-BLG-390L
HN Peg
MOA-2009-BLG-387L
CoRoT-11
2MASS J04414489+2301513
CoRoT-13
OG

KeyboardInterrupt: 

### slower attempt to asynchronously get the planets directly filtered from NEA  

In [6]:
# table upload not possible for NEA :(
# "DALQueryError: ORA-01795: maximum number of expressions in a list is 1000" restricts us, but means we can launch 14 async queries
qstr = f"""
SELECT * FROM pscomppars WHERE hostname IN ({",".join(f"'{id}'" for id in alternate_ids_single["id"][0:1000])})
"""
print(qstr)
all_matching_planets = nasa.run_sync(qstr)
all_matching_planets


SELECT * FROM pscomppars WHERE hostname IN ('TIC 459832522','AP J14153968+1910558','GALAH 150210005801171','GJ 541','HIP 69673','PLX 3242','LSPM J1415+1910','NLTT 36756','TYC 1472-1436-1','ASCC  870215','USNO-B1.0 1091-00221874','* alf Boo','*  16 Boo','AAVSO 1411+19','AG+19 1335','BD+19  2777','CSV 101433','Ci 20  843','FK5  526','GC 19242','GCRV  8341','GEN# +1.00124897','HD 124897','CNS5 3517','HIC  69673','HR  5340','IRAS 14133+1925','IRC +20270','JP11  2486','LFT 1084','LHS    48','LTT 14184','N30 3229','NAME Arcturus','NSV  6603','PM 14134+1927','PM 14134-1927','PMC 90-93   376','PPM 130442','RAFGL 1693','ROT  2044','SAO 100944','SKY# 26028','SRS  30526','SV* ZI  1054','TD1 17351','UBV   12551','UBV M  20076','USNO 850','YPAC  52','[HFE83] 1018','uvby98 100124897','2MASS J14153968+1910558','PLX 3242.00','GAT 1341','** HDS 2003A','WDS J14157+1911A','CCDM J14157+1911A','WEB 12132','TIC 245873777','AP J04355524+1630331','GALAH 141102003801353','GES J04355530+1630309','GJ 171.1 A','

<DALResultsTable length=0>
objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
                           ...                                             
 object   object   object  ...     float64           object        int32   
-------- ------- --------- ... --------------- ----------------- ----------

In [5]:
for i,id in enumerate(alternate_ids_single["id"]):
	if "'" in id:
		print(i, id)



4676 NAME Luyten's star
86977 NAME Barnard's star
93764 NAME Kapteyn's star
93768 NAME Kapteyn's
213903 NAME Wachmann's Flare Star
249999 NAME Innes' star


In [7]:
def segmentation_slices(length, size=1000):
	slices = []
	start = 0
	end = size
	while end < length:
		slices.append(slice(start, end))
		start += size
		end += size
	else:
		slices.pop()
		slices.append(slice(start - size, -1))
	return slices

In [8]:
simbad.maxrec

50000

In [9]:
simbad.hardlimit

2000000

In [10]:
nasa.maxrec
# 

DALServiceError: Default limit not exposed by the service

In [34]:
nasa.hardlimit

DALServiceError: Hard limit not exposed by the service

In [43]:
pscomppars

<DALResultsTable length=6107>
objectid       pl_name        ... pl_ndispec
                              ...           
 object         object        ...   int32   
-------- -------------------- ... ----------
  3.2390        Kepler-1167 b ...          0
  3.1444        Kepler-1740 b ...          0
  3.4135        Kepler-1581 b ...          0
   3.659         Kepler-644 b ...          0
  3.1575        Kepler-1752 b ...          0
  3.2547         Kepler-280 c ...          0
  3.3361        Kepler-1208 b ...          0
  3.3494         Kepler-263 c ...          0
  3.2492        Kepler-1101 b ...          0
     ...                  ... ...        ...
 3.20638         CD-35 2722 b ...          0
 3.11355           HD 86226 b ...          0
 3.20499       HD 143811 AB b ...          1
 3.20644 KMT-2024-BLG-1005L b ...          0
 3.20643            NGTS-34 b ...          0
 3.11415        OGLE-TR-113 b ...          0
  3.1846          Kepler-76 b ...          0
 3.11808            WASP-

In [38]:
seg_slices = segmentation_slices(len(alternate_ids_single["id"]),1000)[0:10]
jobs = []
for i, seg_slice in enumerate(seg_slices):
	jobs.append(nasa.submit_job(f"""
SELECT * FROM pscomppars WHERE hostname IN ({",".join(f"'{id}'" for id in alternate_ids_single["id"][seg_slice])})
    """))
	print(f"SUBMITTED JOB {i}")

print("ALL QUERIES SUBMITTED")

for job in jobs:
	job.run()
	
print("ALL QUERRIES RUN")

for i, job in enumerate(jobs):
	job.wait()
	print(f"job {i:<5} awaited")

results = [job.fetch_result() for job in jobs]
results

# 10 jobs take around 2 mins
# so all jobs will take around an hour (?)

SUBMITTED JOB 0
SUBMITTED JOB 1
SUBMITTED JOB 2
SUBMITTED JOB 3
SUBMITTED JOB 4
SUBMITTED JOB 5
SUBMITTED JOB 6
SUBMITTED JOB 7
SUBMITTED JOB 8
SUBMITTED JOB 9
ALL QUERIES SUBMITTED
ALL QUERRIES RUN
job 0     awaited
job 1     awaited
job 2     awaited
job 3     awaited
job 4     awaited
job 5     awaited
job 6     awaited
job 7     awaited
job 8     awaited
job 9     awaited


[<DALResultsTable length=0>
 objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
                            ...                                             
  object   object   object  ...     float64           object        int32   
 -------- ------- --------- ... --------------- ----------------- ----------,
 <DALResultsTable length=0>
 objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
                            ...                                             
  object   object   object  ...     float64           object        int32   
 -------- ------- --------- ... --------------- ----------------- ----------,
 <DALResultsTable length=0>
 objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
                            ...                                             
  object   object   object  ...     float64           object        int32   
 -------- ------- --------- ... --------------- ----------------- -

In [48]:
from concurrent.futures import ThreadPoolExecutor, as_completed
MAX_CONCURRENT = 100  # server stops connection at 100 simultaneous queries

seg_slices = segmentation_slices(len(alternate_ids_single["id"]),990) # true max is 1000 but ocassionally still terminates

def run_query(query):
    return nasa.run_sync(query)  # async supposedly has 1-2 sec overhead

queries = [f"""SELECT * FROM pscomppars WHERE hostname IN ({",".join(f"'{id}'" for id in alternate_ids_single["id"][s])})""" for s in seg_slices]

results_ = []
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as executor:
    futures = {executor.submit(run_query, q): q for q in queries}
    for f in as_completed(futures):
        q = futures[f]
        try:
            result = f.result()
            results_.append(result)
        except Exception as e:
            print(f"Query failed for {q}: {e}")
            results_.append(None)
			
results_
# takes only 10s for 10 queries!
# 60s for 100 queries!!
# server stops connection at >100 queries
# all 270 queries take around 3.5 minutes, so probably better to work with on-device pscomppars

Query failed for SELECT * FROM pscomppars WHERE hostname IN ('2MASS J17073139-4814538','Gaia DR1 5938742333487153280','Gaia DR2 5938742337829520256','TIC 125115082','UCAC4 211-140720','Gaia DR3 5938815416650641664','2MASS J17051728-4750526','PM J17052-4750','Gaia DR2 5938815416650641664','','TIC 37058215','Gaia DR3 5940966100090146560','2MASS J16404931-4831589','UCAC4 208-129041','PM J16408-4831','Gaia DR2 5940966100090146560','TIC 8380346','HIP 80559','Gaia DR3 5941374534288441728','CD-49 10671','CF 14857','CPD-49  9320','HD 330856','HIC  80559','CNS5 4027','2MASS J16265040-4936221','SSTGLMC G334.4852-00.4047','TYC 8320-1813-1','UCAC4 202-124363','PM J16268-4936','Gaia DR1 5941374529963393152','Gaia DR2 5941374534288441728','TIC 216840997','GJ 618.4','HIP 80229','Gaia DR3 5941478335062267392','PLX 3708','CSI-48-16190','Ci 20  983','HIC  80229','L  338-152','LHS  3185','LTT  6528','CNS5 4007','NLTT 42594','[RHG95]  2575','2MASS J16224099-4839198','PLX 3708.00','TYC 8316-3930-1','GJ 955

[<DALResultsTable length=0>
 objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
                            ...                                             
  object   object   object  ...     float64           object        int32   
 -------- ------- --------- ... --------------- ----------------- ----------,
 <DALResultsTable length=1>
 objectid   pl_name   ... pl_ndispec
                      ...           
  object     object   ...   int32   
 -------- ----------- ... ----------
  3.11042 HD 102365 b ...          0,
 <DALResultsTable length=0>
 objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
                            ...                                             
  object   object   object  ...     float64           object        int32   
 -------- ------- --------- ... --------------- ----------------- ----------,
 <DALResultsTable length=0>
 objectid pl_name pl_letter ... pl_angsepsymerr pl_angsep_reflink pl_ndispec
       